[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Prepared Statements


## What you will be able to do

Say what a prepared statement is, find the ones the server is holding for you right now, and say
which driver put them there. Turn the behavior up, down and off: `prepare_threshold`,
`prepared_max` and `prepare=True` in psycopg, `statement_cache_size` and `conn.prepare()` in
asyncpg. Measure what preparing actually saves, which is less than most people expect and for a
reason worth knowing. And recognize the two errors that come from a plan the server kept after the
schema moved under it, including why one of them only appears sometimes.


## The idea

### The problem

Neither driver told you it was doing this. Both of them quietly ask the server to remember the
parsed, planned form of queries you repeat, under a name, and both of them reuse that name
afterward. It is a good optimization and it is invisible until the day it is not: an `ALTER TABLE`
from a migration, a `SELECT *`, and a query that has run ten thousand times raises
`cached plan must not change result type` on a line nobody touched.

The second way it goes wrong is a connection pooler. PgBouncer in transaction mode hands your next
transaction a different server connection, which has never heard of the name your driver is about
to use.

### What a prepared statement is

Two round trips split apart. Normally the server parses your SQL, plans it, and runs it. `PREPARE`
does the first two and keeps the result under a name; `EXECUTE` does the third, as often as you
like, with different parameters each time. The parameters are sent separately, which is also why
neither driver has ever had to escape a value for you.

### Why it works that way

A plan is only valid for the schema it was planned against. PostgreSQL notices when that is no
longer true, and rather than guess, it refuses. The refusal is the safe behavior: a stale plan
returning the wrong number of columns would be worse.

### Where this shows up

Any long-lived connection, which is to say any pool, which is to say every service. It is also the
reason so much advice about PostgreSQL ends with "and set `statement_cache_size=0`" without saying
what that is for.

### What this notebook covers

Where to look: `pg_prepared_statements`. What each driver does by default, and every knob for
changing it. What preparing saves, measured. Then the two cached-plan errors, the missing-name
error a pooler causes, and the silent retry that makes the first error look random.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    print("psycopg prepares a query it has already run", conn.prepare_threshold, "times")

    for run in range(1, 7):
        conn.execute("SELECT count(*) FROM events WHERE id > %s", (run,))
        held = conn.execute("SELECT name, statement FROM pg_prepared_statements",
                            prepare=False).fetchall()          # never prepare the question itself
        print(f"  run {run}: the server is holding {len(held)} plan(s)")

    for name, statement in held:
        print(f"{name}: {statement}")
```

```
psycopg prepares a query it has already run 5 times
  run 1: the server is holding 0 plan(s)
  run 2: the server is holding 0 plan(s)
  run 3: the server is holding 0 plan(s)
  run 4: the server is holding 0 plan(s)
  run 5: the server is holding 0 plan(s)
  run 6: the server is holding 1 plan(s)
_pg3_0: SELECT count(*) FROM events WHERE id > $1
```

Nothing in that program asked for a prepared statement. psycopg counted, and on the sixth run it
prepared the query under a name it made up. The last line is worth reading twice: the statement the
server is holding says `$1`, not `%s`. That is what psycopg has been sending all along.


## Setup

Eleven imports, both drivers, the server, and three helpers.

- `psycopg` and `asyncpg` are the two drivers, with `errors` and `exceptions` for their classes
- `asyncio` runs the asyncpg half and `time` measures what preparing saves
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

`plans` and `plans_for` list what the server is holding, one for each driver. `against` turns a
timing into a band rather than a number, because the gap measured over a local socket is small
enough that the exact ratio says more about this machine than about preparing.


In [1]:
import asyncio
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from asyncpg import exceptions
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def plans(conn):
    """What the server holds for a psycopg connection, without preparing the question itself."""
    return [name for (name,) in conn.execute(                       # prepare=False exempts this one
        "SELECT name FROM pg_prepared_statements ORDER BY name", prepare=False).fetchall()]


async def plans_for(conn):
    """The same for an asyncpg connection, which has no way to exempt the question."""
    return [record["name"] for record in
            await conn.fetch("SELECT name FROM pg_prepared_statements ORDER BY name")]


def against(baseline, measured):
    """A band rather than a number, because a ratio this small is a property of the machine."""
    ratio = baseline / measured
    if ratio < 1.15:
        return "about the same"
    return "a little faster" if ratio < 2 else "much faster"


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### Looking at what the server holds

`pg_prepared_statements` is a view of the current session, and only the current session. Another
connection cannot see yours, which is the first thing to know about it:


In [2]:
one = psycopg.connect("dbname=guide", autocommit=True)
two = psycopg.connect("dbname=guide", autocommit=True)

for _ in range(6):
    one.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",))

print("what the first connection holds: ", plans(one))
print("what the second one can see:     ", plans(two))


what the first connection holds:  ['_pg3_0']
what the second one can see:      []


So a prepared statement belongs to a session. Close the connection and it is gone, and there is
nothing to clean up.

### Turning psycopg up, down and off

Three settings, and one of them is per call:


In [3]:
print("the defaults: prepare_threshold =", one.prepare_threshold,
      "| prepared_max =", one.prepared_max)

eager = psycopg.connect("dbname=guide", autocommit=True)
eager.execute("SELECT count(*) FROM events", prepare=True)          # this one, right now
print("with prepare=True on the first run: ", plans(eager))

never = psycopg.connect("dbname=guide", autocommit=True)
never.prepare_threshold = None                                      # never, on this connection
for _ in range(8):
    never.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",))
print("with prepare_threshold = None:      ", plans(never))


the defaults: prepare_threshold = 5 | prepared_max = 100
with prepare=True on the first run:  ['_pg3_0']
with prepare_threshold = None:       []


`prepare=True` prepares one statement immediately, `prepare=False` exempts one statement forever,
which is what the helper uses, and `prepare_threshold = None` switches the whole thing off for that
connection.

`prepared_max` caps how many psycopg will keep track of, discarding the least recently used:


In [4]:
capped = psycopg.connect("dbname=guide", autocommit=True)
capped.prepared_max = 3
capped.prepare_threshold = 0                                        # prepare on the first run

for number in range(6):
    capped.execute(f"SELECT count(*) + {number} FROM events")       # six different queries

print("six queries, prepared_max = 3, the server holds:", plans(capped))


six queries, prepared_max = 3, the server holds: ['_pg3_3', '_pg3_4', '_pg3_5']


Three names, not six. The cap exists because a program that builds SQL strings in a loop would
otherwise fill the server's memory with plans it will never use again.

### What asyncpg does instead

asyncpg has no threshold. It prepares everything, from the first run, always:


In [5]:
conn = await asyncpg.connect(database="guide")

await conn.fetchval("SELECT count(*) FROM events WHERE kind = $1", "click")
print("after one query, asyncpg holds:", await plans_for(conn))


after one query, asyncpg holds: ['__asyncpg_stmt_1__', '__asyncpg_stmt_2__']


Two names after one query, and the second one is the query that asked the question. There is no
`prepare=False` in asyncpg, so the introspection prepares itself, which is a small joke and also an
accurate picture of what this driver does with everything.

The cache is sized rather than switched:


In [6]:
print("statement_cache_size defaults to 100")

off = await asyncpg.connect(database="guide", statement_cache_size=0)
await off.fetchval("SELECT count(*) FROM events WHERE kind = $1", "click")
print("with statement_cache_size=0:", await plans_for(off))


statement_cache_size defaults to 100
with statement_cache_size=0: []


Empty, including the introspection, because nothing is being kept. `statement_cache_size=0` is the
setting every PgBouncer instruction ends with, and the Common errors section below shows what it is
protecting you from.

You can also name one yourself, which gives you an object rather than a setting:


In [7]:
statement = await conn.prepare("SELECT count(*) FROM events WHERE kind = $1")

print("the object:    ", type(statement).__name__)
print("what it returns:", [(a.name, a.type.name) for a in statement.get_attributes()])
print("run it:        ", await statement.fetchval("click"), await statement.fetchval("view"))


the object:     PreparedStatement
what it returns: [('count', 'int8')]
run it:         1666 1667


`get_attributes` is asyncpg telling you what the server said the result looks like, before a single
row has been read. That is the same information the server is protecting when it refuses a stale
plan.

### What preparing actually saves

The honest answer, measured twice on two different kinds of query:


In [8]:
async def timed(sql, cache, runs):
    conn = await asyncpg.connect(database="guide", statement_cache_size=cache)
    await conn.fetch(sql, 1)                                        # warm the connection itself
    start = time.perf_counter()
    for number in range(runs):
        await conn.fetch(sql, 1 + number % 3)
    taken = time.perf_counter() - start
    await conn.close()
    return taken


SIMPLE = "SELECT count(*) FROM events WHERE id > $1"
JOINED = ("SELECT a.kind, count(*) FROM events a "
          "JOIN events b ON b.id = a.id + $1 "
          "JOIN events c ON c.id = b.id + 1 "
          "WHERE a.payload @> '{\"size\": 3}' AND b.kind <> c.kind GROUP BY a.kind")

for label, sql, runs in (("a one line query", SIMPLE, 2000), ("a three table join", JOINED, 200)):
    prepared = await timed(sql, 100, runs)
    parsed = await timed(sql, 0, runs)
    print(f"{label}, {runs} times: prepared is", against(parsed, prepared))


a one line query, 2000 times: prepared is a little faster
a three table join, 200 times: prepared is about the same


Preparing saves parsing and planning, and nothing else. So it helps most where parsing and planning
are a large share of the work, which is the cheap query, and least where the query itself is
expensive, which is the join. That is the opposite of the intuition that complicated queries benefit
most.

Two things shrink the gap here further, and both are worth saying out loud: a local Unix socket is
the shortest round trip there is, and `count(*)` over five thousand rows is real work. Against a
server a millisecond away, running a query that returns in microseconds, the saving is larger. Like
**Pipeline Mode**, this is a measurement that understates itself on purpose, because understating is
the honest direction when the machine is this favorable.

### When to reach for which

| What you want | psycopg | asyncpg |
|---|---|---|
| the default | prepare after 5 runs of the same text | prepare everything, from the first run |
| see what is held | `pg_prepared_statements`, this session only | the same view |
| prepare one statement now | `execute(sql, params, prepare=True)` | `await conn.prepare(sql)` |
| never prepare one statement | `prepare=False` on that call | nothing per call |
| never prepare anything | `conn.prepare_threshold = None` | `statement_cache_size=0` |
| limit how many are kept | `conn.prepared_max` | `statement_cache_size=N` |
| behind PgBouncer in transaction mode | `prepare_threshold = None` | `statement_cache_size=0` |

The default is to leave both alone. They are on for a reason and the settings exist for two
situations: a pooler that breaks the assumption, and a program that generates SQL text it will never
repeat.

### A reporting query that survives a migration, finished

The habit that prevents the whole class of error, written as a function and then proved against an
`ALTER TABLE` running underneath it.


In [9]:
async def summarize(conn):
    """Named columns, so the result type cannot change under a plan the server is holding."""
    rows = await conn.fetch(
        "SELECT kind, count(*) AS n FROM events GROUP BY kind ORDER BY kind")
    return [(row["kind"], row["n"]) for row in rows]


reporter = await asyncpg.connect(database="guide")
migrator = await asyncpg.connect(database="guide")

print("before:", await summarize(reporter))

await migrator.execute("ALTER TABLE events ADD COLUMN source text")
async with reporter.transaction():                                  # the strictest case
    print("after: ", await summarize(reporter))

await migrator.execute("ALTER TABLE events DROP COLUMN source")
print("the plan the server holds names its columns, so adding one changed nothing")


before: [('click', 1666), ('purchase', 1667), ('view', 1667)]
after:  [('click', 1666), ('purchase', 1667), ('view', 1667)]
the plan the server holds names its columns, so adding one changed nothing


The same `ALTER TABLE`, the same transaction, the same driver: the only difference from the failure
in Common errors below is that this query names its columns. `SELECT *` means "whatever the columns
are", and adding one changes the answer to that question.

### Where each part came from

| In the reporting query | What it relies on | The section that showed it |
|---|---|---|
| `await conn.fetch(...)` | a list of `Record` | **asyncpg** |
| named columns | a result type that an `ALTER` cannot change | What preparing actually saves |
| the second connection | a migration running elsewhere | Looking at what the server holds |
| `async with reporter.transaction()` | no silent re-prepare to hide the problem | No error, most of the time |
| the plan being reused at all | asyncpg preparing from the first run | What asyncpg does instead |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/12-prepared-statements-solutions.ipynb).

**1.** Run one query six times with psycopg and print what the server holds after each run.


In [10]:
# your code here


**2.** Prepare a psycopg query on its first run instead of its sixth.


In [11]:
# your code here


**3.** Open an asyncpg connection that never prepares anything, and show the server holds nothing.


In [12]:
# your code here


**4.** Name an asyncpg prepared statement yourself and run it with two different parameters.


In [13]:
# your code here


**5.** Show that one connection cannot see another connection's prepared statements.


In [14]:
# your code here


**6.** Cap psycopg at two prepared statements and run four different queries.


In [15]:
# your code here


## Common errors

### psycopg.errors.FeatureNotSupported: cached plan must not change result type


In [16]:
reader = psycopg.connect("dbname=guide", autocommit=True)
migrator_sync = psycopg.connect("dbname=guide", autocommit=True)

reader.execute("DROP TABLE IF EXISTS reports")
reader.execute("CREATE TABLE reports (id int, note text)")
reader.execute("INSERT INTO reports VALUES (1, 'the first one')")

for _ in range(6):                                                  # past the threshold
    reader.execute("SELECT * FROM reports").fetchall()

migrator_sync.execute("ALTER TABLE reports ADD COLUMN extra int")   # somebody else's migration
reader.execute("SELECT * FROM reports").fetchall()


FeatureNotSupported: cached plan must not change result type

The plan the server is holding says this query returns two columns. The table now has three.
PostgreSQL will not guess, so it refuses, and the traceback points at a line that has been correct
for months.

Two fixes, and they are not equivalent:


In [17]:
print("name the columns, and adding one is invisible:")
print("  ", reader.execute("SELECT id, note FROM reports").fetchall())

reader.prepare_threshold = None                                     # the blunt fix
print("or stop preparing, and pay for the parse every time:")
print("  ", reader.execute("SELECT * FROM reports").fetchall())


name the columns, and adding one is invisible:
   [(1, 'the first one')]
or stop preparing, and pay for the parse every time:
   [(1, 'the first one', None)]


Naming the columns fixes the cause. Turning preparing off hides it, and costs you the optimization
everywhere on that connection. Reach for the first one.

### asyncpg.exceptions.InvalidCachedStatementError: cached statement plan is invalid


In [18]:
watcher = await asyncpg.connect(database="guide")
changer = await asyncpg.connect(database="guide")

await watcher.execute("DROP TABLE IF EXISTS reports")
await watcher.execute("CREATE TABLE reports (id int, note text)")
await watcher.execute("INSERT INTO reports VALUES (1, 'the first one')")
await watcher.fetch("SELECT * FROM reports")                        # cached from the first run

await changer.execute("ALTER TABLE reports ADD COLUMN extra int")

async with watcher.transaction():
    await watcher.fetch("SELECT * FROM reports")


InvalidCachedStatementError: cached statement plan is invalid due to a database schema or configuration change

The same cause, a different class, and one extra condition: the transaction block. It matters, and
the next section is about why.

Note the `ALTER TABLE` had to come from another connection. A schema change needs a lock no open
transaction will give it, so running both halves on one connection does not fail, it waits.

### No error, most of the time


In [19]:
await changer.execute("ALTER TABLE reports ADD COLUMN another int")

print("outside a transaction:", await watcher.fetch("SELECT * FROM reports"))

async with watcher.transaction():
    print("and now inside one: ", await watcher.fetch("SELECT * FROM reports"))


outside a transaction: [<Record id=1 note='the first one' extra=None another=None>]
and now inside one:  [<Record id=1 note='the first one' extra=None another=None>]


The first line is the same query that raised a moment ago, and it works. asyncpg caught the error,
threw its cached plan away, prepared the query again and ran it, all without telling you. The second
line then works too, because the cache is fresh by the time it runs.

Inside a transaction the retry is not available: the failed statement has already put the
transaction into an error state, so there is nothing to retry into. That is the whole difference,
and it is why this error arrives looking random. It is not random. It is the same error every time,
surfaced only when a transaction was open.

### asyncpg.exceptions.InvalidSQLStatementNameError: prepared statement does not exist


In [20]:
pooled = await asyncpg.connect(database="guide")
await pooled.fetchval("SELECT count(*) FROM events WHERE kind = $1", "click")

await pooled.execute("DEALLOCATE ALL")                              # what a pooler does for you

await pooled.fetchval("SELECT count(*) FROM events WHERE kind = $1", "click")


InvalidSQLStatementNameError: prepared statement "__asyncpg_stmt_9__" does not exist
HINT:  
NOTE: pgbouncer with pool_mode set to "transaction" or
"statement" does not support prepared statements properly.
You have two options:

* if you are using pgbouncer for connection pooling to a
  single server, switch to the connection pool functionality
  provided by asyncpg, it is a much better option for this
  purpose;

* if you have no option of avoiding the use of pgbouncer,
  then you can set statement_cache_size to 0 when creating
  the asyncpg connection object.


`DEALLOCATE ALL` throws away every prepared statement on the session. The driver has no way to know,
so the next query asks for a name the server no longer has.

Read the hint on that exception rather than skipping it: asyncpg names PgBouncer and tells you what
to do about it, because this is almost always what has happened. In transaction pooling mode the
pooler hands your next transaction a different server connection, and the effect is exactly this.

psycopg survives the literal statement, because it watches for it in the text you send. It does not
survive the pooler, for the same reason asyncpg does not: nobody told it:


In [21]:
survivor = psycopg.connect("dbname=guide", autocommit=True)
for _ in range(6):
    survivor.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",))
print("before:", plans(survivor))

survivor.execute("DEALLOCATE ALL")
print("after: ", plans(survivor), "and the next query still works:",
      survivor.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",)).fetchone())


before: ['_pg3_0']
after:  [] and the next query still works: (1666,)


In [22]:
for connection in (one, two, eager, never, capped, reader, migrator_sync, survivor):
    connection.close()
for connection in (conn, off, reporter, migrator, watcher, changer, pooled):
    await connection.close()
print("connections closed")


connections closed


## Recap

- A prepared statement is a parse and a plan the server keeps under a name. It belongs to one
  session, it is listed in `pg_prepared_statements`, and closing the connection disposes of it.
- psycopg prepares a query after `prepare_threshold` runs of the same text, five by default, keeps
  `prepared_max` of them, and takes `prepare=True` or `prepare=False` on a single call.
- asyncpg prepares everything from the first run. `statement_cache_size=0` turns that off and
  `conn.prepare()` gives you the statement as an object.
- Preparing saves parsing and planning, so the cheap query gains most and the expensive join gains
  least. Over a local socket the whole saving is small.
- A schema change invalidates a held plan. psycopg raises `FeatureNotSupported: cached plan must not
  change result type`, asyncpg raises `InvalidCachedStatementError`, and naming your columns instead
  of `SELECT *` prevents both.
- asyncpg re-prepares and retries silently outside a transaction, which is why the error looks
  intermittent when it is not.
- A pooler in transaction mode breaks the name, which is what `statement_cache_size=0` and
  `prepare_threshold = None` are for.


## What is next

**Connection Pools** is the object that owns the connections all of this has been happening on: how
many to have, why the answer comes from the database server's cores rather than from your traffic,
and what a pool does when the server restarts underneath it.


---

&#8592; **Previous:** [asyncpg](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/11-asyncpg.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
